# RollingLDA execution

## 1. Setup

In [1]:
# Cell 1: Imports
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr, PackageNotInstalledError
from rpy2.robjects.conversion import localconverter

import pyLDAvis
from IPython.display import display as ipy_display

In [2]:
# Cell 2: Set directories and columns

CHECKPOINT_DIR = Path("checkpoints")
OUT_DIR = Path("outputs_rlda_fitting")
OUT_DIR.mkdir(exist_ok=True)

BRAND_COL = "designer"
TOKEN_COL = "noun_tokens"
DATE_COL = "_date"
DOC_ID_COL = "doc_id"

In [3]:
# Cell 3: Set parameters

K = None
INIT_DATE = None
CHUNKS = "year" 
MEMORY = "2 years"
NUM_ITERATIONS = 200
RANDOM_STATE = 42

# Vocabulary thresholds used by rollinglda
VOCAB_ABS = 5
VOCAB_REL = 0
VOCAB_FALLBACK = 100
DOC_ABS = 0

INITIAL_MODEL_TYPE = "ldaprototype"
N_PROTOTYPE_RUNS = 100

## 2. Check R packages

In [4]:
# Cell 4: Install R package
ro.r('install.packages("rollinglda", repos="https://cloud.r-project.org")')

R callback write-console: trying URL 'https://cloud.r-project.org/bin/macosx/big-sur-arm64/contrib/4.5/rollinglda_0.1.4.tgz'
  
R callback write-console: Content type 'application/x-gzip'  
R callback write-console:  length 434627 bytes (424 KB)
  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R callback write-console: =  
R


The downloaded binary packages are in
	/var/folders/_p/f1mg9b290kx6bf1ytjkd60qm0000gn/T//Rtmpkmm5dD/downloaded_packages


In [5]:
# Cell 5: Import R packages

for pkg in ["rollinglda", "lda", "ldaPrototype"]:
    try:
        importr(pkg)
        print(f"OK: {pkg}")
    except PackageNotInstalledError:
        print(f"MISSING: {pkg} — run in R: install.packages('{pkg}')")
        raise

rollinglda = importr("rollinglda")
lda_pkg = importr("lda")
ldaprototype = importr("ldaPrototype")

ro.r("library(rollinglda)")
ro.r("library(lda)")
ro.r("library(ldaPrototype)")


OK: rollinglda
OK: lda
OK: ldaPrototype


## 3. Load data

In [6]:
# Cell 4: Load data and vocab, set up dataframe for R

data_ckpt = CHECKPOINT_DIR / "df_with_noun_tokens.pkl"
if not data_ckpt.exists():
    raise FileNotFoundError(f"{data_ckpt} not found — run notebook 2.1 first.")

df = pd.read_pickle(data_ckpt).copy()
df[DATE_COL] = pd.to_datetime(df[DATE_COL])

# Load chosen vocabulary notebook 2.2.
vocab_ckpt = CHECKPOINT_DIR / "chosen_vocab.pkl"
if not vocab_ckpt.exists():
    raise FileNotFoundError(f"{vocab_ckpt} not found — run notebook 2.2 first.")
with open(vocab_ckpt, "rb") as f:
    vc = pickle.load(f)
chosen_vocab_set = set(vc["chosen_vocab"]["kept_terms"])
print(f"Loaded chosen vocab: {len(chosen_vocab_set)} terms")

# Reuse K and INIT_DATE notebook 2.3
rlda_ckpt = CHECKPOINT_DIR / "rlda_init_vocab.pkl"
if rlda_ckpt.exists():
    with open(rlda_ckpt, "rb") as f:
        rc = pickle.load(f)
    if K is None:
        K = int(rc["CHOSEN_K_RLDA"])
    if INIT_DATE is None:
        INIT_DATE = pd.to_datetime(rc["INIT_DATE"])

if K is None or INIT_DATE is None:
    raise ValueError("Run notebook 2.3 first")

# Keep only usable rows.
df = df[df[TOKEN_COL].apply(lambda x: isinstance(x, list))].copy()
df = df.sort_values(DATE_COL).reset_index(drop=True)

# Pre-filter tokens to chosen vocab before passing to R.
df["_tokens_filtered"] = df[TOKEN_COL].apply(
    lambda toks: [t for t in toks if t.lower() in chosen_vocab_set]
)

# Ensure unique document ids (R requirement)
if DOC_ID_COL not in df.columns:
    df[DOC_ID_COL] = [f"doc_{i}" for i in range(len(df))]
else:
    df[DOC_ID_COL] = df[DOC_ID_COL].astype(str)
    if df[DOC_ID_COL].duplicated().any():
        df[DOC_ID_COL] = [f"{doc_id}_{i}" for i, doc_id in enumerate(df[DOC_ID_COL])]

n_empty = (df["_tokens_filtered"].apply(len) == 0).sum()
print(f"Loaded {len(df):,} documents ({n_empty} empty after vocab filter)")
print(f"Date range: {df[DATE_COL].min().date()} → {df[DATE_COL].max().date()}")
print(f"K={K}, INIT_DATE={INIT_DATE.date()}, chunks={CHUNKS}, memory={MEMORY}")

Loaded chosen vocab: 6249 terms
Loaded 6,501 documents (0 empty after vocab filter)
Date range: 1999-09-12 → 2014-06-12
K=10, INIT_DATE=2003-12-31, chunks=year, memory=2 years


## 4. Convert Python tokens to R format

In [7]:
# Cell 5: Convert data to R format

def make_r_texts_and_dates(frame: pd.DataFrame, token_col: str, date_col: str, id_col: str):
    names = frame[id_col].astype(str).tolist()

    r_texts = ro.ListVector({
        name: ro.StrVector([str(tok) for tok in toks if isinstance(tok, str) and tok])
        for name, toks in zip(names, frame[token_col])
    })

    r_dates = ro.r["as.Date"](ro.StrVector(frame[date_col].dt.strftime("%Y-%m-%d").tolist()))
    r_dates.names = ro.StrVector(names)

    return r_texts, r_dates

r_texts, r_dates = make_r_texts_and_dates(df, "_tokens_filtered", DATE_COL, DOC_ID_COL)

print(f"Converted {len(r_texts)} tokenized texts to R (vocab-filtered)")
print(f"First doc id: {list(r_texts.names)[0]}")

Converted 6501 tokenized texts to R (vocab-filtered)
First doc id: 69


## 5. Fit RollingLDA

In [8]:
# Cell 6: fit rlda model

rlda_model = rollinglda.RollingLDA(
    texts=r_texts,
    dates=r_dates,
    chunks=CHUNKS,
    memory=MEMORY,
    init=INIT_DATE.strftime("%Y-%m-%d"),
    K=ro.IntVector([int(K)]),
    alpha=ro.FloatVector([1 / K]),
    eta=ro.FloatVector([1 / K]),
    type=INITIAL_MODEL_TYPE,
    id="rlda-official-package",
    seeds=ro.IntVector([int(RANDOM_STATE)]),
    **{
        "num.iterations": ro.IntVector([int(NUM_ITERATIONS)]),
        "vocab.abs": ro.IntVector([int(VOCAB_ABS)]),
        "vocab.rel": ro.IntVector([int(VOCAB_REL)]),
        "vocab.fallback": ro.IntVector([int(VOCAB_FALLBACK)]),
        "doc.abs": ro.IntVector([int(DOC_ABS)]),
        "n":              ro.IntVector([int(N_PROTOTYPE_RUNS)]),
    }
)

print(rlda_model)

R callback write-console: Fitting LDAPrototype as initial model.
  
R callback write-console: No seeds given or length of given seeds differs from number of replications: sample seeds. Sampled seeds can be obtained via getJob().
  
R callback write-console: Exporting objects to package env on master for mode: local
  
R callback write-console: Fitting Chunk 1/11.
  
R callback write-console: Fitting Chunk 2/11.
  
R callback write-console: Fitting Chunk 3/11.
  
R callback write-console: Fitting Chunk 4/11.
  
R callback write-console: Fitting Chunk 5/11.
  
R callback write-console: Fitting Chunk 6/11.
  
R callback write-console: Fitting Chunk 7/11.
  
R callback write-console: Fitting Chunk 8/11.
  
R callback write-console: Fitting Chunk 9/11.
  
R callback write-console: Fitting Chunk 10/11.
  
R callback write-console: Fitting Chunk 11/11.
  
R callback write-console: Compute topic matrix.
  


RollingLDA Object named "rlda-official-package" with elements
"id", "lda", "docs", "dates", "vocab", "chunks", "param"
 12 Chunks with Texts from 1999-09-12 to 2014-06-12
 vocab.abs: 5, vocab.rel: 0, vocab.fallback: 100, doc.abs: 0

LDA Object with element(s)
"param", "assignments", "topics", "document_sums"
 6501 Texts with mean length of 47.38 Tokens
 2749 different Words
 K: 10, alpha: 0.1, eta: 0.1, num.iterations: 200




## 6. Extract chunk metadata, vocabulary, and LDA internals

In [9]:
# Cell 7: Get model outputs and convert to Python

getChunks = ro.r["getChunks"]
getLDA = ro.r["getLDA"]
getVocab = ro.r["getVocab"]

r_chunks = getChunks(rlda_model)
r_lda = getLDA(rlda_model)
r_vocab = getVocab(rlda_model)

with localconverter(ro.default_converter + pandas2ri.converter):
    chunks_df = ro.conversion.rpy2py(r_chunks)

vocab = list(r_vocab)

# LDA object fields: assignments, topics, document_sums
with localconverter(ro.default_converter + pandas2ri.converter):
    topics = np.array(r_lda.rx2("topics"))
    document_sums = np.array(r_lda.rx2("document_sums"))


# Use 1e - 12 for nimerical stability, avoid division by 0
phi = topics / np.clip(topics.sum(axis=1, keepdims=True), 1e-12, None)

print(chunks_df.head())
print(f"topics shape: {topics.shape} K x V")
print(f"document_sums shape: {document_sums.shape} K x D")
print(f"vocab size: {len(vocab):,}")
print(f"phi shape: {phi.shape}")


   chunk.id  start.date  end.date   memory    n  n.discarded    n.memory  \
1         0     10846.0   12344.0      NaN  936            0 -2147483648   
2         1     12435.0   12701.0  11688.0  236            0         440   
3         2     12817.0   13065.0  12053.0  282            0         458   
4         3     13182.0   13429.0  12418.0  334            0         518   
5         4     13545.0   13794.0  12784.0  421            0         616   

   n.vocab  
1     1590  
2     1621  
3     1662  
4     1722  
5     1800  
topics shape: (10, 2749) K x V
document_sums shape: (10, 6501) K x D
vocab size: 2,749
phi shape: (10, 2749)


/Users/zoeoggel/Data Science/Thesis/.ths/lib/python3.13/site-packages/rpy2/robjects/conversion.py:28: DeprecationWarning: The use of rpy2py in module rpy2.robjects.conversion is deprecated. Use rpy2.robjects.conversion.get_conversion() instead of rpy2.robjects.conversion.converter.
  warnings.warn(


## 7. Save outputs

In [10]:
# Cell 8: Save outputs (full R object, chunk schedule, topic-word counts, doc-topic proportions)

# full R object
ro.globalenv["roll_lda"] = rlda_model
ro.r(f"saveRDS(roll_lda, file='{str((OUT_DIR / 'rollinglda_official_model.rds').as_posix())}')")

# chunk schedule
chunks_df.to_csv(OUT_DIR / "rollinglda_chunks.csv", index=False)

# final topic-word counts
topic_word_df = pd.DataFrame(topics, columns=vocab)
topic_word_df.insert(0, "topic_id", range(topics.shape[0]))
topic_word_df.to_csv(OUT_DIR / "rollinglda_final_topic_word_counts.csv", index=False)

# document-topic proportions
doc_topic_counts = document_sums.T
row_sums = doc_topic_counts.sum(axis=1, keepdims=True)
doc_topic = (doc_topic_counts + (1 / K)) / np.clip(row_sums + K * (1 / K), 1e-12, None)

doc_topic_df = pd.DataFrame(doc_topic, columns=[f"topic_{k}" for k in range(K)])
doc_topic_df.insert(0, DOC_ID_COL, df[DOC_ID_COL].values[:len(doc_topic_df)])
doc_topic_df.insert(1, DATE_COL, df[DATE_COL].values[:len(doc_topic_df)])
doc_topic_df.to_csv(OUT_DIR / "rollinglda_doc_topic.csv", index=False)

print(f"Saved outputs to {OUT_DIR.resolve()}")


Saved outputs to /Users/zoeoggel/Data Science/Thesis/Thesis-Git/outputs_rlda_fitting


## 8. Inspect top words

In [11]:
# Cell 9: Get top words per topic and save

N_TOP_WORDS = 15

phi = topics / np.clip(topics.sum(axis=1, keepdims=True), 1e-12, None)
rows = []
for k in range(K):
    top_idx = np.argsort(phi[k])[::-1][:N_TOP_WORDS]
    rows.append({
        "topic_id": k,
        "top_words": ", ".join(vocab[i] for i in top_idx)
    })

# Borrowed from notebook 2.3
def get_top_words_per_topic(components: np.ndarray, feature_names: list, n: int = N_TOP_WORDS, rel: np.ndarray = None) -> pd.DataFrame:
    rows = []
    for idx, topic in enumerate(components):
        top_idx = rel[idx].argsort()[:-n - 1:-1] if rel is not None else topic.argsort()[:-n - 1:-1]
        words   = [feature_names[i] for i in top_idx]
        rows.append({"topic_id": idx, "top_words": ", ".join(words), "top_words_list": words})
    return pd.DataFrame(rows)

top_words_df = pd.DataFrame(rows)
top_words_df.to_csv(OUT_DIR / "rollinglda_top_words.csv", index=False)
top_words_df


,topic_id,top_words
0,0,"print, jacket, shirt, leather, pant, white, sk..."
1,1,"jacket, coat, skirt, leather, pant, short, whi..."
2,2,"coat, skirt, jacket, silk, gown, leather, wool..."
3,3,"skirt, white, jacket, leather, pant, silk, shi..."
4,4,"lace, gown, print, white, jacket, chiffon, coa..."
5,5,"leather, jacket, coat, skirt, print, big, pant..."
6,6,"print, coat, jacket, skirt, leather, today, sh..."
7,7,"print, skirt, jacket, silk, white, floral, lea..."
8,8,"jacket, leather, skirt, coat, body, white, wor..."
9,9,"print, skirt, white, jacket, short, silk, lace..."


In [12]:
# Cell 10: Top words by LDAvis relevance (λ=0.4)
# relevance(w,k) = λ·log p(w|k) + (1−λ)·log [p(w|k) / p(w)]

LAMBDA_VIS       = 0.4
N_TOP_WORDS_REL  = 15

p_w_corpus = topics.sum(axis=0).astype(float)
p_w_corpus /= p_w_corpus.sum()

rel = (
    LAMBDA_VIS       * np.log(phi.clip(min=1e-12))
    + (1 - LAMBDA_VIS) * np.log((phi / p_w_corpus.clip(min=1e-12)).clip(min=1e-12))
)

rel_words_df = get_top_words_per_topic(phi, vocab, n=N_TOP_WORDS_REL, rel=rel)
rel_words_df = rel_words_df.rename(columns={
    "top_words": f"top_words_lambda{int(LAMBDA_VIS * 10):02d}",
    "top_words_list": f"top_words_list_lambda{int(LAMBDA_VIS * 10):02d}",
})

raw_df = get_top_words_per_topic(phi, vocab, n=N_TOP_WORDS_REL)
rel_words_df["top_words_raw"] = raw_df["top_words"]

rel_words_df.to_csv(OUT_DIR / f"rollinglda_top_words_lambda{int(LAMBDA_VIS * 10):02d}.csv", index=False)
rel_words_df

,topic_id,top_words_lambda04,top_words_list_lambda04,top_words_raw
0,0,"neon, print, stripe, jean, sporty, fun, bright...","[neon, print, stripe, jean, sporty, fun, brigh...","print, jacket, shirt, leather, pant, white, sk..."
1,1,"jacket, coat, pant, boot, skinny, lamb, leathe...","[jacket, coat, pant, boot, skinny, lamb, leath...","jacket, coat, skirt, leather, pant, short, whi..."
2,2,"cashmere, coat, wool, tweed, silk, gown, skirt...","[cashmere, coat, wool, tweed, silk, gown, skir...","coat, skirt, jacket, silk, gown, leather, wool..."
3,3,"skirt, white, shirt, waist, masculine, importa...","[skirt, white, shirt, waist, masculine, import...","skirt, white, jacket, leather, pant, silk, shi..."
4,4,"lace, gown, carpet, chiffon, actress, siren, r...","[lace, gown, carpet, chiffon, actress, siren, ...","lace, gown, print, white, jacket, chiffon, coa..."
5,5,"leather, coat, accessory, bag, belt, menswear,...","[leather, coat, accessory, bag, belt, menswear...","leather, jacket, coat, skirt, print, big, pant..."
6,6,"doll, catwalk, stage, polka, dot, baby, scene,...","[doll, catwalk, stage, polka, dot, baby, scene...","print, coat, jacket, skirt, leather, today, sh..."
7,7,"print, floral, strapless, flower, pink, rhines...","[print, floral, strapless, flower, pink, rhine...","print, skirt, jacket, silk, white, floral, lea..."
8,8,"work, complex, technical, body, material, form...","[work, complex, technical, body, material, for...","jacket, leather, skirt, coat, body, white, wor..."
9,9,"summer, lace, spring, cotton, print, innocent,...","[summer, lace, spring, cotton, print, innocent...","print, skirt, white, jacket, short, silk, lace..."


In [13]:
# Cell 9: Get top words by LDAvis relevance

LAMBDA_VIS = 0.4
N_TOP_WORDS_REL = 15

p_w_corpus = topics.sum(axis=0).astype(float)
p_w_corpus /= p_w_corpus.sum()

rel = (
    LAMBDA_VIS * np.log(phi.clip(min=1e-12))
    + (1 - LAMBDA_VIS) * np.log((phi / p_w_corpus.clip(min=1e-12)).clip(min=1e-12))
)

rows = []
for k in range(K):
    top_rel = np.argsort(rel[k])[::-1][:N_TOP_WORDS_REL]
    top_raw = np.argsort(phi[k])[::-1][:N_TOP_WORDS_REL]
    rows.append({
        "topic_id": k,
        f"top_words_lambda{int(LAMBDA_VIS * 10):02d}": ", ".join(vocab[i] for i in top_rel),
        "top_words_raw":                               ", ".join(vocab[i] for i in top_raw),
    })

rel_words_df = pd.DataFrame(rows)
rel_words_df.to_csv(OUT_DIR / f"rollinglda_top_words_lambda{int(LAMBDA_VIS * 10):02d}.csv", index=False)
rel_words_df

,topic_id,top_words_lambda04,top_words_raw
0,0,"neon, print, stripe, jean, sporty, fun, bright...","print, jacket, shirt, leather, pant, white, sk..."
1,1,"jacket, coat, pant, boot, skinny, lamb, leathe...","jacket, coat, skirt, leather, pant, short, whi..."
2,2,"cashmere, coat, wool, tweed, silk, gown, skirt...","coat, skirt, jacket, silk, gown, leather, wool..."
3,3,"skirt, white, shirt, waist, masculine, importa...","skirt, white, jacket, leather, pant, silk, shi..."
4,4,"lace, gown, carpet, chiffon, actress, siren, r...","lace, gown, print, white, jacket, chiffon, coa..."
5,5,"leather, coat, accessory, bag, belt, menswear,...","leather, jacket, coat, skirt, print, big, pant..."
6,6,"doll, catwalk, stage, polka, dot, baby, scene,...","print, coat, jacket, skirt, leather, today, sh..."
7,7,"print, floral, strapless, flower, pink, rhines...","print, skirt, jacket, silk, white, floral, lea..."
8,8,"work, complex, technical, body, material, form...","jacket, leather, skirt, coat, body, white, wor..."
9,9,"summer, lace, spring, cotton, print, innocent,...","print, skirt, white, jacket, short, silk, lace..."


In [ ]:
# Cell 10: LDAvis for a specific year

# Year for inspection
YEAR = 2005

year_mask = pd.to_datetime(df[DATE_COL]).dt.year == YEAR
df_year   = df[year_mask].reset_index(drop=True)

vocab_index_y = {w: i for i, w in enumerate(vocab)}
X_year = np.zeros((len(df_year), len(vocab)), dtype=np.int32)
for d, toks in enumerate(df_year["_tokens_filtered"]):
    if not isinstance(toks, list):
        continue
    for tok in toks:
        idx = vocab_index_y.get(str(tok).lower())
        if idx is not None:
            X_year[d, idx] += 1

doc_topic_y = X_year @ phi.T
doc_topic_y = doc_topic_y / doc_topic_y.sum(axis=1, keepdims=True).clip(min=1e-12)

doc_lengths_y = X_year.sum(axis=1).astype(int)
term_freq_y   = X_year.sum(axis=0).astype(int)

vis_year = pyLDAvis.prepare(
    topic_term_dists=phi,
    doc_topic_dists=doc_topic_y,
    doc_lengths=doc_lengths_y,
    vocab=vocab,
    term_frequency=term_freq_y,
    lambda_step=0.1,
    sort_topics=False,
)

# out_path = OUT_DIR / f"rlda_ldavis_{YEAR}.html"
# pyLDAvis.save_html(vis_year, str(out_path))
print(f"LDAvis {YEAR}: {len(df_year)} docs -> {out_path}")
ipy_display(pyLDAvis.display(vis_year))

/Users/zoeoggel/Data Science/Thesis/.ths/lib/python3.13/site-packages/pandas/core/internals/blocks.py:347: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)
/Users/zoeoggel/Data Science/Thesis/.ths/lib/python3.13/site-packages/pandas/core/internals/blocks.py:347: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)
/Users/zoeoggel/Data Science/Thesis/.ths/lib/python3.13/site-packages/pandas/core/internals/blocks.py:347: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)


NameError: name 'out_path' is not defined